In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!tar -xzvf /content/drive/MyDrive/Datasets/hwu.tar.gz

hwu/
hwu/categories.json
hwu/train_5.csv
hwu/train_10.csv
hwu/val.csv
hwu/test.csv
hwu/train.csv


## Bước 0:

In [3]:
import pandas as pd

df_train = pd.read_csv('/content/hwu/train.csv', header=None, names=['text', 'intent'])
df_val = pd.read_csv('/content/hwu/val.csv', header=None, names=['text', 'intent'])
df_test = pd.read_csv('/content/hwu/test.csv', header=None, names=['text', 'intent'])

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
df_train.head()

Train shape: (8955, 2)
Validation shape: (1077, 2)
Test shape: (1077, 2)


,text,intent
0,text,category
1,what alarms do i have set right now,alarm_query
2,checkout today alarm of meeting,alarm_query
3,report alarm settings,alarm_query
4,see see for me the alarms that you have set to...,alarm_query


In [4]:
x_train = df_train['text']
y_train = df_train['intent']
x_val = df_val['text']
y_val = df_val['intent']
x_test = df_test['text']
y_test = df_test['intent']

In [5]:
from sklearn.preprocessing import LabelEncoder

all_intents = pd.concat([df_train['intent'], df_val['intent'], df_test['intent']]).unique()

label_encoder = LabelEncoder()
label_encoder.fit(all_intents)

y_train = label_encoder.transform(df_train['intent'])
y_val = label_encoder.transform(df_val['intent'])
y_test = label_encoder.transform(df_test['intent'])

num_classes = len(label_encoder.classes_)
print(f"Số lượng classes: {num_classes}")
print("Các classes:", label_encoder.classes_)

Số lượng classes: 65
Các classes: ['alarm_query' 'alarm_remove' 'alarm_set' 'audio_volume_down'
 'audio_volume_mute' 'audio_volume_up' 'calendar_query' 'calendar_remove'
 'calendar_set' 'category' 'cooking_recipe' 'datetime_convert'
 'datetime_query' 'email_addcontact' 'email_query' 'email_querycontact'
 'email_sendemail' 'general_affirm' 'general_commandstop'
 'general_confirm' 'general_dontcare' 'general_explain' 'general_joke'
 'general_negate' 'general_praise' 'general_quirky' 'general_repeat'
 'iot_cleaning' 'iot_coffee' 'iot_hue_lightchange' 'iot_hue_lightdim'
 'iot_hue_lightoff' 'iot_hue_lighton' 'iot_hue_lightup' 'iot_wemo_off'
 'iot_wemo_on' 'lists_createoradd' 'lists_query' 'lists_remove'
 'music_likeness' 'music_query' 'music_settings' 'news_query'
 'play_audiobook' 'play_game' 'play_music' 'play_podcasts' 'play_radio'
 'qa_currency' 'qa_definition' 'qa_factoid' 'qa_maths' 'qa_stock'
 'recommendation_events' 'recommendation_locations'
 'recommendation_movies' 'social_post' '

## Nhiệm vụ 1:

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000)
)

tfidf_lr_pipeline.fit(df_train['text'], y_train)

y_pred_tfidf = tfidf_lr_pipeline.predict(df_test['text'])
print("Classification Report (TF-IDF + Logistic Regression):")
print(classification_report(y_test, y_pred_tfidf))

tfidf_f1_macro = classification_report(y_test, y_pred_tfidf, output_dict=True)['macro avg']['f1-score']
print(f"TF-IDF + Logistic Regression F1-score (Macro): {tfidf_f1_macro:.4f}")

Classification Report (TF-IDF + Logistic Regression):
              precision    recall  f1-score   support

           0       0.90      0.95      0.92        19
           1       1.00      0.73      0.84        11
           2       0.81      0.89      0.85        19
           3       1.00      0.75      0.86         8
           4       0.92      0.80      0.86        15
           5       0.93      1.00      0.96        13
           6       0.48      0.53      0.50        19
           7       0.89      0.89      0.89        19
           8       0.82      0.74      0.78        19
           9       0.00      0.00      0.00         1
          10       0.59      0.68      0.63        19
          11       0.67      0.75      0.71         8
          12       0.74      0.89      0.81        19
          13       0.78      0.88      0.82         8
          14       0.83      0.79      0.81        19
          15       0.92      0.63      0.75        19
          16       0.77    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

## Nhiệm vụ 2:

In [7]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 68.9 MB/s eta 0:00:00


In [8]:
import numpy as np
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
# 1. Huấn luyện mô hình Word2Vec trên dữ liệu text của bạn
sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

print(f"Kích thước vector Word2Vec: {w2v_model.vector_size}")
print(f"Số lượng từ trong từ điển Word2Vec: {len(w2v_model.wv)}")

# 2. Viết hàm để chuyển mỗi câu thành vector trung bình
def sentence_to_avg_vector(text, model):
    tokens = text.split()
    vectors = []
    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

# 3. Tạo dữ liệu train/val/test X_train_avg, X_val_avg, X_test_avg
X_train_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_train['text']])
X_val_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_val['text']])
X_test_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_test['text']])

print("X_train_avg shape:", X_train_avg.shape)
print("X_test_avg shape:", X_test_avg.shape)

# 4. Xây dựng mô hình Sequential của Keras
model_w2v_dense = Sequential([
    Dense(256, activation='relu', input_shape=(w2v_model.vector_size,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model_w2v_dense.compile(loss="sparse_categorical_crossentropy",optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
model_w2v_dense.summary()

# 5. Compile, huấn luyện và đánh giá mô hình
print("Huấn luyện mô hình Word2Vec + Dense:")
history_w2v_dense = model_w2v_dense.fit(
    X_train_avg, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_avg, y_val),
    verbose=1
)

print("\nĐánh giá mô hình trên tập test:")
loss_w2v_dense, accuracy_w2v_dense = model_w2v_dense.evaluate(X_test_avg, y_test)
print(f"Test Loss: {loss_w2v_dense:.4f}")
print(f"Test Accuracy: {accuracy_w2v_dense:.4f}")

# Dự đoán để tính F1-score macro
y_pred_w2v_dense = np.argmax(model_w2v_dense.predict(X_test_avg), axis=-1)
w2v_dense_f1_macro = classification_report(y_test, y_pred_w2v_dense, output_dict=True)['macro avg']['f1-score']
print(f"Word2Vec + Dense F1-score (Macro): {w2v_dense_f1_macro:.4f}")
print(f"Word2Vec + Dense Test Loss: {loss_w2v_dense:.4f}")

Kích thước vector Word2Vec: 100
Số lượng từ trong từ điển Word2Vec: 4467
X_train_avg shape: (8955, 100)
X_test_avg shape: (1077, 100)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        25,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,673 (268.25 KB)

 Trainable params: 67,905 (265.25 KB)

 Non-trainable params: 768 (3.00 KB)

Huấn luyện mô hình Word2Vec + Dense:
Epoch 1/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.0711 - loss: 4.1788 - val_accuracy: 0.0371 - val_loss: 4.1007
Epoch 2/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1791 - loss: 3.2370 - val_accuracy: 0.0687 - val_loss: 4.0702
Epoch 3/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2309 - loss: 2.9582 - val_accuracy: 0.1987 - val_loss: 3.1244
Epoch 4/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2615 - loss: 2.7248 - val_accuracy: 0.1894 - val_loss: 3.2992
Epoch 5/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2809 - loss: 2.6216 - val_accuracy: 0.0966 - val_loss: 4.6976
Epoch 6/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3007 - loss: 2.5467 - val_accuracy: 0.1291 - val_loss: 4.7991
Epoch 7/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3170 - loss: 2.4927 - val_accuracy: 0.2108 - val_loss: 3.3626
Epoch 8/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Nhiệm vụ 3: Mô hình Nâng cao (Embedding Pre-trained + LSTM)

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import SpatialDropout1D, Bidirectional
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau

# 1. Tiền xử lý cho mô hình chuỗi

# a. Tokenizer: Tạo vocab và chuyển text thành chuỗi chỉ số
tokenizer = Tokenizer(num_words=None, oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])

train_sequences = tokenizer.texts_to_sequences(df_train['text'])
val_sequences = tokenizer.texts_to_sequences(df_val['text'])
test_sequences = tokenizer.texts_to_sequences(df_test['text'])

# b. Padding: Đảm bảo các chuỗi có cùng độ dài
max_len = 50
X_train_pad = pad_sequences(train_sequences, maxlen=max_len, padding='post')
X_val_pad = pad_sequences(val_sequences, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(test_sequences, maxlen=max_len, padding='post')

print("Shape sau padding:")
print(f"X_train_pad: {X_train_pad.shape}")
print(f"X_test_pad: {X_test_pad.shape}")

# 2. Tạo ma trận trọng số cho Embedding Layer từ Word2Vec
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Tỷ lệ words có trong Word2Vec: {np.sum(embedding_matrix.sum(axis=1) > 0) / vocab_size:.2%}")

# 3. Xây dựng mô hình Sequential với LSTM
lstm_model_pretrained = Sequential([
    Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix], # Khởi tạo trọng số
    input_length=max_len,
    trainable=False # Đóng băng lớp Embedding
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
  ])

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)

lstm_model_pretrained.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
lstm_model_pretrained.summary()


print("Huấn luyện mô hình Pre-trained Embedding + LSTM:")
history_pretrained = lstm_model_pretrained.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("\nĐánh giá mô hình trên tập test:")
loss_pretrained, accuracy_pretrained = lstm_model_pretrained.evaluate(X_test_pad, y_test)
print(f"Test Loss: {loss_pretrained:.4f}")
print(f"Test Accuracy: {accuracy_pretrained:.4f}")

# Dự đoán để tính F1-score macro
y_pred_pretrained = np.argmax(lstm_model_pretrained.predict(X_test_pad), axis=-1)
pretrained_f1_macro = classification_report(y_test, y_pred_pretrained, output_dict=True)['macro avg']['f1-score']
print(f"Pre-trained Embedding + LSTM F1-score (Macro): {pretrained_f1_macro:.4f}")
print(f"Pre-trained Embedding + LSTM Test Loss: {loss_pretrained:.4f}")

Shape sau padding:
X_train_pad: (8955, 50)
X_test_pad: (1077, 50)
Embedding matrix shape: (4265, 100)
Tỷ lệ words có trong Word2Vec: 22.60%


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       426,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 426,500 (1.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 426,500 (1.63 MB)

Huấn luyện mô hình Pre-trained Embedding + LSTM:
Epoch 1/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 39s 162ms/step - accuracy: 0.0175 - loss: 4.1562 - val_accuracy: 0.0353 - val_loss: 4.0010 - learning_rate: 0.0010
Epoch 2/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 31s 162ms/step - accuracy: 0.0330 - loss: 4.0269 - val_accuracy: 0.0446 - val_loss: 3.8464 - learning_rate: 0.0010
Epoch 3/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 22s 158ms/step - accuracy: 0.0481 - loss: 3.9044 - val_accuracy: 0.0548 - val_loss: 3.8021 - learning_rate: 0.0010
Epoch 4/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 166ms/step - accuracy: 0.0523 - loss: 3.8538 - val_accuracy: 0.0641 - val_loss: 3.7461 - learning_rate: 0.0010
Epoch 5/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 23s 162ms/step - accuracy: 0.0610 - loss: 3.8087 - val_accuracy: 0.0613 - val_loss: 3.7540 - learning_rate: 0.0010
Epoch 6/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 40s 155ms/step - accuracy: 0.0551 - loss: 3.7865 - val_accuracy: 0.0576 - val_loss: 3.6841 - learning_rate: 0.0010
Epoch 7/20
140/140 ━━

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Nhiệm vụ 4: Mô hình Nâng cao (Embedding học từ đầu + LSTM)

In [10]:
# Dữ liệu đã được tiền xử lý (tokenized, padded) từ nhiệm vụ 3
# 1. Xây dựng mô hình
lstm_model_scratch = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=200,  # Tăng chiều embedding
        input_length=max_len
    ),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

lstm_model_scratch.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model_scratch.summary()

# 2. Huấn luyện với EarlyStopping và đánh giá mô hình
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Huấn luyện mô hình Scratch Embedding + LSTM:")
history_scratch = lstm_model_scratch.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val_pad, y_val),
    callbacks=[early_stopping],
    verbose=1
)

print("\nĐánh giá mô hình trên tập test:")
loss_scratch, accuracy_scratch = lstm_model_scratch.evaluate(X_test_pad, y_test)
print(f"Test Loss: {loss_scratch:.4f}")
print(f"Test Accuracy: {accuracy_scratch:.4f}")

# Dự đoán để tính F1-score macro
y_pred_scratch = np.argmax(lstm_model_scratch.predict(X_test_pad), axis=-1)
scratch_f1_macro = classification_report(y_test, y_pred_scratch, output_dict=True)['macro avg']['f1-score']
print(f"Scratch Embedding + LSTM F1-score (Macro): {scratch_f1_macro:.4f}")
print(f"Scratch Embedding + LSTM Test Loss: {loss_scratch:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Huấn luyện mô hình Scratch Embedding + LSTM:
Epoch 1/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 194s 644ms/step - accuracy: 0.1664 - loss: 3.5873 - val_accuracy: 0.7539 - val_loss: 2.6184
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 176s 627ms/step - accuracy: 0.7648 - loss: 0.9598 - val_accuracy: 0.8477 - val_loss: 0.6487
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 176s 627ms/step - accuracy: 0.8796 - loss: 0.4706 - val_accuracy: 0.8589 - val_loss: 0.5347
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 176s 630ms/step - accuracy: 0.9216 - loss: 0.2934 - val_accuracy: 0.8793 - val_loss: 0.5009
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 174s 619ms/step - accuracy: 0.9390 - loss: 0.2256 - val_accuracy: 0.8700 - val_loss: 0.5439
Epoch 6/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 176s 629ms/step - accuracy: 0.9514 - loss: 0.1730 - val_accuracy: 0.8812 - val_loss: 0.5353
Epoch 7/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 175s 626ms/step - accuracy: 0.9559 - loss: 0.1584 - val_accuracy: 0.8756 - val_loss: 0.5632
Epoch 8/10
280/280 ━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Nhiệm vụ 5: Đánh giá, So sánh và Phân tích


In [11]:
# Tạo bảng tổng hợp kết quả
import pandas as pd

results_df = pd.DataFrame({
    'Pipeline': [
        'TF-IDF + Logistic Regression',
        'Word2Vec (Avg) + Dense',
        'Embedding (Pre-trained) + LSTM',
        'Embedding (Scratch) + LSTM'
    ],
    'F1-score (Macro)': [
        tfidf_f1_macro,
        w2v_dense_f1_macro,
        pretrained_f1_macro,
        scratch_f1_macro
    ],
    'Test Loss': [
        'N/A',
        loss_w2v_dense,
        loss_pretrained,
        loss_scratch
    ]
})

print("Bảng tổng hợp kết quả các mô hình:")
print(results_df.to_string(index=False))

# Tìm mô hình tốt nhất theo F1-score macro
best_model = results_df.loc[results_df['F1-score (Macro)'].idxmax()]
print(f"\nMô hình tốt nhất: {best_model['Pipeline']}")
print(f"Với F1-score (Macro): {best_model['F1-score (Macro)']:.4f}")

Bảng tổng hợp kết quả các mô hình:
                      Pipeline  F1-score (Macro) Test Loss
  TF-IDF + Logistic Regression          0.822567       N/A
        Word2Vec (Avg) + Dense          0.187212  3.633471
Embedding (Pre-trained) + LSTM          0.046148  3.452877
    Embedding (Scratch) + LSTM          0.836721  0.603244

Mô hình tốt nhất: Embedding (Scratch) + LSTM
Với F1-score (Macro): 0.8367


In [12]:
print("Phân tích một vài câu từ tập test:")

# Lấy 5 câu ngẫu nhiên từ test set để phân tích
import random
random.seed(42)
sample_indices = random.sample(range(len(df_test)), min(5, len(df_test)))

for i, idx in enumerate(sample_indices, 1):
    text = df_test.iloc[idx]['text']
    true_label = y_test[idx]
    true_intent = label_encoder.inverse_transform([true_label])[0]

    print(f"\nCâu {i}: '{text}'")
    print(f"Intent thực tế: {true_intent}")

    # Dự đoán từ các mô hình
    # TF-IDF + LR
    tfidf_pred = tfidf_lr_pipeline.predict([text])[0]
    tfidf_intent = label_encoder.inverse_transform([tfidf_pred])[0]

    # Word2Vec + Dense
    text_avg = sentence_to_avg_vector(text, w2v_model).reshape(1, -1)
    w2v_pred = np.argmax(model_w2v_dense.predict(text_avg, verbose=0), axis=-1)[0]
    w2v_intent = label_encoder.inverse_transform([w2v_pred])[0]

    # LSTM Pre-trained
    text_seq = tokenizer.texts_to_sequences([text])
    text_pad = pad_sequences(text_seq, maxlen=max_len, padding='post')
    lstm_pred = np.argmax(lstm_model_pretrained.predict(text_pad, verbose=0), axis=-1)[0]
    lstm_intent = label_encoder.inverse_transform([lstm_pred])[0]

    # LSTM Scratch
    lstm2_pred = np.argmax(lstm_model_scratch.predict(text_pad, verbose=0), axis=-1)[0]
    lstm2_intent = label_encoder.inverse_transform([lstm2_pred])[0]

    print("Dự đoán từ các mô hình:")
    print(f"  TF-IDF + LR:          {tfidf_intent} {'✓' if tfidf_pred == true_label else '✗'}")
    print(f"  Word2Vec + Dense:     {w2v_intent} {'✓' if w2v_pred == true_label else '✗'}")
    print(f"  LSTM Pre-trained:     {lstm_intent} {'✓' if lstm_pred == true_label else '✗'}")
    print(f"  LSTM Scratch:         {lstm2_intent} {'✓' if lstm2_pred == true_label else '✗'}")

    # Phân tích kết quả
    correct_models = []
    if tfidf_pred == true_label:
        correct_models.append("TF-IDF+LR")
    if w2v_pred == true_label:
        correct_models.append("W2V+Dense")
    if lstm_pred == true_label:
        correct_models.append("LSTM-Pre")
    if lstm2_pred == true_label:
        correct_models.append("LSTM-Scratch")

    print(f"  Mô hình đúng: {', '.join(correct_models) if correct_models else 'Không có'}")


Phân tích một vài câu từ tập test:

Câu 1: 'is this my sisters cellphone number'
Intent thực tế: email_querycontact
Dự đoán từ các mô hình:
  TF-IDF + LR:          email_querycontact ✓
  Word2Vec + Dense:     transport_traffic ✗
  LSTM Pre-trained:     qa_factoid ✗
  LSTM Scratch:         email_querycontact ✓
  Mô hình đúng: TF-IDF+LR, LSTM-Scratch

Câu 2: 'please talk softer'
Intent thực tế: audio_volume_down
Dự đoán từ các mô hình:
  TF-IDF + LR:          audio_volume_mute ✗
  Word2Vec + Dense:     audio_volume_mute ✗
  LSTM Pre-trained:     general_affirm ✗
  LSTM Scratch:         audio_volume_down ✓
  Mô hình đúng: LSTM-Scratch

Câu 3: 'create a new shopping list'
Intent thực tế: lists_createoradd
Dự đoán từ các mô hình:
  TF-IDF + LR:          lists_createoradd ✓
  Word2Vec + Dense:     email_query ✗
  LSTM Pre-trained:     lists_createoradd ✓
  LSTM Scratch:         lists_createoradd ✓
  Mô hình đúng: TF-IDF+LR, LSTM-Pre, LSTM-Scratch

Câu 4: 'reduce brightness'
Intent thực tế: i